In [ ]:
!pip install datasets faker reportlab pdf2image pillow pandas

In [ ]:
import os
import json
import random
import io
import pandas as pd
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4, LETTER
from reportlab.lib import colors
from reportlab.lib.utils import ImageReader
from PIL import Image, ImageDraw, ImageFont

# Check for required libraries and install if not found.
try:
    from datasets import load_dataset
    from faker import Faker
except ImportError:
    os.system("pip install -q datasets faker reportlab pdf2image pillow")
    from datasets import load_dataset
    from faker import Faker

# Base directory for the output files and Faker locale setup.
OUTPUT_DIR = "dataset_master_final_v3"
fake = Faker('de_DE')

# Map template placeholders to specific data generation functions.
PLACEHOLDER_MAP = {
    '[NAME]': lambda: fake.first_name() + " " + fake.last_name(),
    '[PERSON_NAME]': lambda: fake.name(),
    '[PERSON]': lambda: fake.name(),
    '[COMPANY_NAME]': lambda: fake.company(),
    '[ORG]': lambda: fake.company(),
    '[COMPANY]': lambda: fake.company(),
    '[CUSTOMER_ID]': lambda: str(random.randint(100000, 999999)),
    '[IBAN_CODE]': lambda: fake.iban(),
    '[IBAN]': lambda: fake.iban(),
    '[STREET_ADDRESS]': lambda: fake.street_address(),
    '[LOCATION]': lambda: fake.city(),
    '[ADDRESS]': lambda: fake.address().replace('\n', ', '),
    '[EMAIL_ADDRESS]': lambda: fake.email(),
    '[EMAIL]': lambda: fake.email(),
    '[PHONE_NUMBER]': lambda: fake.phone_number(),
    '[DATE]': lambda: fake.date(),
    '[TIME]': lambda: fake.time()
}

# Define strategies for different document categories.
DOC_STRATEGIES = {
    'TABLE': ['Invoice', 'Bill', 'Receipt', 'Bank Statement', 'Financial Statement', 'Tax Assessment Notice', 'Purchase Order', 'Payment Confirmation'],
    'CONTRACT': ['Employment Contract', 'Loan Agreement', 'NDA', 'Shareholder Agreement', 'Real Estate Loan Agreement', 'Settlement Agreement'],
    'FORM': ['Credit Application', 'Health Insurance Claim Form', 'Insurance Claim Form', 'Loan Application', 'Tax Return', 'Credit Card Application']
}

# Select strategy based on document type string matching.
def get_doc_strategy(doc_type):
    for strategy, types in DOC_STRATEGIES.items():
        if any(t in doc_type for t in types): return strategy
    return 'CLEAN_REPORT'

# List of document types to be excluded from the generation process.
EXCLUDED_TYPES = ['BAI Format', 'CSV', 'EDI', 'FIX Protocol', 'FpML', 'ISDA Definition', 'IT support ticket', 'MT940', 'SWIFT Message', 'XBRL', 'Financial Data Feed', 'Cryptocurrency Transaction Report', 'Customer support conversational log']

# Class for generating synthetic scan-like images.
class ScanGenerator:
    
    def __init__(self):
        try:
            self.font_bold = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf", 18)
            self.font_reg = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf", 14)
            self.font_mono = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationMono-Regular.ttf", 14)
        except:
            self.font_bold = ImageFont.load_default()
            self.font_reg = ImageFont.load_default()
            self.font_mono = ImageFont.load_default()
    
    def create_background(self, width, height, color=(255, 255, 255)): return Image.new('RGB', (width, height), color)
    
    def create_id_card(self, name, id_num):
        width, height = 400, 250
        img = self.create_background(width, height, color=(220, 220, 220))
        draw = ImageDraw.Draw(img)
        draw.rectangle([20, 60, 100, 180], fill=(100, 100, 100), outline=(50,50,50))
        draw.text((120, 30), "ID_DOCUMENT", fill=(0,0,0), font=self.font_bold)
        draw.text((120, 80), "Name:", fill=(50,50,50), font=self.font_reg)
        w_name = len(name) * 10
        gt_data = [{"label": "person_name", "text": name, "rel_bbox": [120, 100, 120 + w_name, 118]}]
        draw.text((120, 100), name, fill=(0,0,0), font=self.font_bold)
        draw.text((120, 140), "ID Number:", fill=(50,50,50), font=self.font_reg)
        draw.text((120, 160), id_num, fill=(0,0,0), font=self.font_mono)
        return img, gt_data

# Main class to handle PDF creation and coordinate normalization.
class DocGenerator:
    def __init__(self, output_path, data_item):
        self.c = canvas.Canvas(output_path, pagesize=random.choice([A4, LETTER]))
        self.width, self.height = self.c._pagesize
        self.doc_type = data_item['document_type']
        self.text_content = self._process_text(data_item)
        self.strategy = get_doc_strategy(self.doc_type)
        self.ground_truth = []
        self.scan_gen = ScanGenerator()
        self.f_main, self.f_bold, self.f_mono = 'Helvetica', 'Helvetica-Bold', 'Courier'
        self.margin_left, self.margin_right = random.randint(45, 60), random.randint(45, 60)
        self.cursor_y = self.height - 50
    
    def _process_text(self, item):
        text = item['generated_text']
        try:
            spans = item['pii_spans']
            if isinstance(spans, str): spans = json.loads(spans)
            spans = sorted(spans, key=lambda x: x['start'], reverse=True)
            for span in spans: text = text[:span['start']] + f"[{span['label'].upper()}]" + text[span['end']:]
            return text
        except: return text
    
    def _add_to_gt(self, label, text, x1, y1_bottom, x2, y2_top):
        y_top = self.height - y2_top
        y_bottom = self.height - y1_bottom
        bbox = [max(0, min(1000, int((x1 / self.width) * 1000))), max(0, min(1000, int((y_top / self.height) * 1000))), max(0, min(1000, int((x2 / self.width) * 1000))), max(0, min(1000, int((y_bottom / self.height) * 1000)))]
        self.ground_truth.append({"label": label, "text": text, "bbox": bbox})
    
    def _log_text_gt(self, label, text, x, y, size, font):
        if not text: return
        w = self.c.stringWidth(text, font, size)
        p = (2, 12, 3, 3) if 'Courier' in font else (1, 3, 2, 2)
        self._add_to_gt(label, text, x - p[0], y - p[3], x + w + p[1], y + size + p[2])
    
    def check_page(self, space):
        if self.cursor_y - space < 50:
            self.c.showPage()
            self.cursor_y = self.height - 50
            return True
        return False
    
    def render_body(self):
        paragraphs = self.text_content.split('\n')
        self.c.setFont(self.f_main, 10)
        for para in paragraphs:
            if not para.strip(): continue
            self.check_page(40)
            words = para.split()
            line_text, line_width, line_gts = "", 0, []
            
            
            for word in words:
                clean_word = word.strip(".,:;()[]")
                bracket_key = f"[{clean_word}]"
                final_word, label = word, None
                
                if bracket_key in PLACEHOLDER_MAP:
                    replacement = PLACEHOLDER_MAP[bracket_key]()
                    if 'NAME' in bracket_key or 'PERSON' in bracket_key: label = 'person_name'
                    elif 'COMPANY' in bracket_key or 'ORG' in bracket_key: label = 'company_name'
                    elif 'IBAN' in bracket_key: label = 'iban'
                    elif 'ADDRESS' in bracket_key or 'LOCATION' in bracket_key: label = 'location'
                    if word.endswith(','): replacement += ','
                    elif word.endswith('.'): replacement += '.'
                    final_word = replacement
                sw = self.c.stringWidth(" ", self.f_main, 10)
                ww = self.c.stringWidth(final_word, self.f_main, 10)
                added_w = ww + (sw if line_text else 0)
                
                if line_width + added_w < (self.width - self.margin_left - self.margin_right):
                    if label: line_gts.append({'label': label, 'text': final_word.strip(".,:"), 'x': line_width + (sw if line_text else 0)})
                    line_text += (" " if line_text else "") + final_word
                    line_width += added_w
                
                else:
                    self.c.drawString(self.margin_left, self.cursor_y, line_text)
                    for item in line_gts: self._log_text_gt(item['label'], item['text'], self.margin_left + item['x'], self.cursor_y, 10, self.f_main)
                    self.cursor_y -= 12
                    self.check_page(40)
                    line_text, line_width, line_gts = final_word, ww, []
                    if label: line_gts.append({'label': label, 'text': final_word.strip(".,:"), 'x': 0})
            
            if line_text:
                self.c.drawString(self.margin_left, self.cursor_y, line_text)
                for item in line_gts: self._log_text_gt(item['label'], item['text'], self.margin_left + item['x'], self.cursor_y, 10, self.f_main)
                self.cursor_y -= 15
    
    def generate(self):
        if self.strategy == 'TABLE': self.render_table()
        elif self.strategy == 'CONTRACT': self.render_contract()
        elif self.strategy == 'FORM': self.render_form()
        else: self.render_report()
        self.c.save()
        return self.ground_truth, self.strategy
    
    def render_table(self):
        self.c.setFont(self.f_bold, 16); self.c.drawString(self.margin_left, self.cursor_y, self.doc_type.upper()); self.cursor_y -= 40
        name = fake.name(); self.c.setFont(self.f_main, 11); self.c.drawString(self.margin_left, self.cursor_y, f"Client: {name}")
        self._log_text_gt('person_name', name, self.margin_left + self.c.stringWidth("Client: ", self.f_main, 11), self.cursor_y, 11, self.f_main)
        self.cursor_y -= 40; self._draw_table_data()
    
    def _draw_table_data(self):
        cols, widths = ["Date", "Party", "IBAN", "Amount"], [60, 140, 150, 80]
        self.c.setFont(self.f_bold, 9); curr_x = self.margin_left
        for i, col in enumerate(cols): self.c.drawString(curr_x, self.cursor_y, col); curr_x += widths[i]
        self.cursor_y -= 15; self.c.setFont(self.f_main, 9)
        for _ in range(random.randint(3, 8)):
            self.check_page(20); curr_x = self.margin_left; d, p, i, b = fake.date(), fake.name(), fake.iban(), f"{random.randint(100,9000)} EUR"
            self.c.drawString(curr_x, self.cursor_y, d); curr_x += widths[0]
            self.c.drawString(curr_x, self.cursor_y, p); self._log_text_gt('person_name', p, curr_x, self.cursor_y, 9, self.f_main); curr_x += widths[1]
            self.c.drawString(curr_x, self.cursor_y, i); self._log_text_gt('iban', i, curr_x, self.cursor_y, 9, self.f_main); curr_x += widths[2]
            self.c.drawString(curr_x, self.cursor_y, b); self.cursor_y -= 15
    # Render contract headers, body content and append a signature field.
    def render_contract(self):
        self.c.setFont(self.f_bold, 16); self.c.drawCentredString(self.width/2, self.cursor_y, self.doc_type.upper()); self.cursor_y -= 50
        self.render_body(); self.check_page(100); self.cursor_y -= 30; name = fake.name()
        self.c.drawString(self.margin_left, self.cursor_y, "Signature:"); self.c.drawString(self.margin_left + 80, self.cursor_y, name)
        self._log_text_gt('person_name', name, self.margin_left + 80, self.cursor_y, 10, self.f_main)
    
    # Layout structured input fields with associated labels for form documents.
    def render_form(self):
        self.c.setFont(self.f_bold, 16); self.c.drawString(self.margin_left, self.cursor_y, self.doc_type.upper()); self.cursor_y -= 40
        fields = [("Name:", 'person_name', fake.name), ("IBAN:", 'iban', fake.iban), ("Address:", 'location', fake.address)]
        for label, pii, func in fields:
            self.check_page(40); self.c.setFont(self.f_main, 10); self.c.drawString(self.margin_left, self.cursor_y, label)
            val = func().replace('\n', ', '); self.c.setFont(self.f_mono, 10); self.c.drawString(self.margin_left + 80, self.cursor_y, val)
            self._log_text_gt(pii, val, self.margin_left + 80, self.cursor_y, 10, self.f_mono); self.cursor_y -= 30
    
    # Draw the layout for a standard report including header and main body text.
    def render_report(self):
        self.c.setFont(self.f_bold, 18); self.c.drawString(self.margin_left, self.cursor_y, self.doc_type); self.cursor_y -= 40
        name = fake.name(); self.c.setFont(self.f_main, 10); self.c.drawString(self.margin_left, self.cursor_y, "Created for: ")
        self.c.setFont(self.f_bold, 10); self.c.drawString(self.margin_left + 80, self.cursor_y, name)
        self._log_text_gt('person_name', name, self.margin_left + 80, self.cursor_y, 10, self.f_bold)
        self.cursor_y -= 30; self.render_body()

# Primary execution function.
def run_generator():
    ds = load_dataset("gretelai/synthetic_pii_finance_multilingual", split="train")
    german_ds = ds.filter(lambda x: x['language'] == 'German')
    final_ds = german_ds.filter(lambda x: x['document_type'] not in EXCLUDED_TYPES)
    split_1 = final_ds.train_test_split(test_size=0.3, seed=42)
    split_2 = split_1['test'].train_test_split(test_size=0.5, seed=42)
    data_splits = {"train": split_1['train'], "val": split_2['train'], "test": split_2['test']}
    for name in data_splits.keys(): os.makedirs(os.path.join(OUTPUT_DIR, name), exist_ok=True)
    for split_name, data in data_splits.items():
        rows = []
        target = os.path.join(OUTPUT_DIR, split_name)
        for i, item in enumerate(data):
            try:
                filepath = os.path.join(target, f"doc_{i}.pdf")
                gen = DocGenerator(filepath, item)
                gt, strat = gen.generate()
                rows.append({"pdf_path": filepath, "document_type": item['document_type'], "strategy": strat, "ground_truth": json.dumps(gt, ensure_ascii=False)})
            except Exception as e: print(f"Error at {split_name} doc {i}: {e}")
        pd.DataFrame(rows).to_csv(os.path.join(target, "metadata.csv"), index=False)

if __name__ == "__main__": run_generator()

In [ ]:
import os
import json
import pandas as pd
import random
from PIL import Image, ImageDraw, ImageFont
from pdf2image import convert_from_path

# Path to the generated dataset folder and metadata file.
INPUT_DIR = "dataset_master_final_v3"
METADATA_FILE = os.path.join(INPUT_DIR, "metadata.csv")

# Directory for storing the visualization results.
OUTPUT_DIR = "visual_check_100"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Process random samples from the dataset to visualize bounding boxes.
def run_mass_visualization():
    if not os.path.exists(METADATA_FILE):
        print(f"Error: metadata.csv not found in {INPUT_DIR}.")
        return

    print("Loading metadata...")
    df = pd.read_csv(METADATA_FILE)

    # Select up to 100 random document entries for visual validation.
    samples = df.sample(100, random_state=42) if len(df) > 100 else df

    print(f"Generating 100 verification images in '{OUTPUT_DIR}'...")
    success_count = 0

    for idx, row in samples.iterrows():
        pdf_path = row['pdf_path']
        strategy = row['strategy']
        gt_data = json.loads(row['ground_truth'])
        base_name = os.path.basename(pdf_path).replace('.pdf', '')
        output_filename = f"{OUTPUT_DIR}/{base_name}_{strategy}.png"

        try:
            # Convert PDF pages to images for annotation.
            images = convert_from_path(pdf_path, dpi=100)
            img = images[0].convert("RGB")
            draw = ImageDraw.Draw(img)
            width, height = img.size

            # Draw bounding boxes based on normalized 0-1000 coordinates.
            for item in gt_data:
                norm_box = item['bbox']
                x1 = (norm_box[0] / 1000) * width
                y1 = (norm_box[1] / 1000) * height
                x2 = (norm_box[2] / 1000) * width
                y2 = (norm_box[3] / 1000) * height
                draw.rectangle([x1, y1, x2, y2], outline="green", width=2)

            # Save the processed image to the output directory.
            img.save(output_filename)
            success_count += 1
            if success_count % 10 == 0:
                print(f"Progress: {success_count} images created.")

        except Exception as e:
            print(f"Error processing {pdf_path}: {e}")

    
    print(f"Done! {success_count} images saved in '{OUTPUT_DIR}'.")
    

if __name__ == "__main__":
    run_mass_visualization()